# Assignment 1 — Data Preprocessing & EDA (Spambase)

Generic `assn1_preprocess()` function — works on any tabular classification dataset. Dataset used here: `Spambase_Dataset.csv`, target column `spam`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.preprocessing import MinMaxScaler, Binarizer

from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

## Assignment 1 — `assn1_preprocess()`\n\nGeneric preprocessing function, broken into **7 numbered steps** so it's easy to walk through in an exam: inspect -> remove duplicates -> handle missing values -> split features/target -> EDA -> scale -> train/test split. Works on **any** dataset — just change `df` and `target_column`.

In [ ]:
def assn1_preprocess(
    df,
    target_column,
    scale=True,
    remove_duplicates=True,
    test_size=0.20,
    random_state=42
):
    """
    ==================================================================
    ASSIGNMENT 1 — GENERIC PREPROCESSING PIPELINE
    ==================================================================
    This ONE function is meant to work on ANY tabular classification
    dataset. You just pass in a dataframe and tell it which column is
    the target (the thing you're trying to predict), and it does all
    the standard "get the data ready for ML" work for you.

    Parameters
    ----------
    df : pandas DataFrame
        The raw dataset, exactly as read from a CSV.
    target_column : str
        Name of the column you want to predict (e.g. "spam").
    scale : bool
        Whether to squash all features into the 0-1 range using
        MinMaxScaler. Almost always True for KNN/NB style algorithms
        because they are sensitive to feature scale.
    remove_duplicates : bool
        Whether to drop exact duplicate rows before training.
    test_size : float
        Fraction of data held out for testing (0.20 = 20%).
    random_state : int
        A "seed" so that the random train/test split is reproducible
        (same split every time you run the notebook).

    Returns
    -------
    dict with keys: df, X, y, X_train, X_test, y_train, y_test, scaler
    """

    # ==============================================================
    # STEP 1: INSPECT THE DATA
    # ==============================================================
    # Before touching anything, always LOOK at the data first. This
    # tells us: how many rows/columns, what types of columns we have,
    # whether anything is obviously missing or broken.
    print("=" * 70)
    print("DATASET INFORMATION")
    print("=" * 70)

    # df.shape -> (number_of_rows, number_of_columns)
    print("\nShape :", df.shape)

    # df.head() shows the first 5 rows so we can eyeball the data
    display(df.head())

    print("\nColumn Names")
    print(df.columns.tolist())

    print()
    # df.info() shows column dtypes (int/float/object) and how many
    # non-null values each column has -> quick way to spot missing data
    df.info()

    print("\nStatistical Summary")
    # df.describe() gives mean/std/min/max/quartiles for numeric columns
    # (and count/unique/top/freq for categorical ones, via include="all")
    display(df.describe(include="all"))

    print("\nMissing Values Before Cleaning")
    # isnull() marks every cell True/False depending on whether it's
    # missing; .sum() adds those up per column -> count of NaNs per column
    print(df.isnull().sum())

    print("\nDuplicate Rows :", df.duplicated().sum())
    # duplicated() flags rows that are IDENTICAL to an earlier row

    # ==============================================================
    # STEP 2: REMOVE DUPLICATES
    # ==============================================================
    # Why remove duplicates? If the same row appears twice, the model
    # effectively "sees" that example twice, giving it more importance
    # than it deserves and slightly biasing training. Also duplicate
    # rows can leak from train into test if we're not careful, which
    # would make our test accuracy look better than it really is.
    if remove_duplicates:
        df = df.drop_duplicates().reset_index(drop=True)
        # reset_index(drop=True) re-numbers the rows 0,1,2,... after
        # dropping some — otherwise the index would have gaps in it.

    # ==============================================================
    # STEP 3: HANDLE MISSING VALUES
    # ==============================================================

    # ---- 3a. Drop rows where the TARGET itself is missing ----
    # If we don't know the true label (spam or not spam) for a row,
    # that row is USELESS for supervised learning — we can't train on
    # it and we can't fairly test on it. So we just remove those rows.
    if df[target_column].isnull().sum() > 0:
        print(f"\nRemoving {df[target_column].isnull().sum()} rows with missing target values")
        df = df.dropna(subset=[target_column]).reset_index(drop=True)

    # ---- 3b. Fill missing FEATURE values ----
    # First, separate columns into numeric vs categorical, because we
    # fill them differently.
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

    # Numeric columns -> fill missing values with the MEDIAN.
    # Why median and not mean? The median is robust to outliers.
    # e.g. if most values are around 10 but one row is 10,000 (an
    # outlier/typo), the mean gets dragged way up, but the median
    # barely moves. So median is the safer "typical value" to plug in.
    for col in numeric_cols:
        if col != target_column:
            df[col] = df[col].fillna(df[col].median())

    # Categorical columns -> fill missing values with the MODE
    # (the most frequently occurring category). This is the natural
    # equivalent of "median" for non-numeric data — you can't average
    # text categories, but you can find the most common one.
    for col in categorical_cols:
        if col != target_column:
            df[col] = df[col].fillna(df[col].mode()[0])

    print("\nMissing Values After Cleaning")
    print(df.isnull().sum())  # should all be 0 now

    # ==============================================================
    # STEP 4: SPLIT FEATURES / TARGET
    # ==============================================================
    # X = everything the model is allowed to look at (the "inputs")
    # y = the thing we're trying to predict (the "answer")
    # This split is required before scaling and train/test splitting,
    # because we never want to accidentally scale or leak the target.
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # ==============================================================
    # STEP 5: EDA (Exploratory Data Analysis) PLOTS
    # ==============================================================
    # EDA just means "look at the data visually before modelling" so
    # we understand what we're working with (balanced classes? skewed
    # features? correlated features?).

    # ---- Class distribution ----
    # A bar count of how many rows belong to each class. If one class
    # has 95% of the rows and the other has 5%, that's an IMBALANCED
    # dataset, and plain accuracy becomes a misleading metric (a lazy
    # model that always predicts the majority class would still score
    # 95% "accuracy" while being useless).
    plt.figure(figsize=(6, 5))
    sns.countplot(x=y)
    plt.title("Class Distribution")
    plt.show()

    # ---- Histograms ----
    # One histogram per numeric column, showing how its values are
    # spread out (normal-shaped? skewed? has outliers?).
    df.hist(figsize=(20, 18), bins=20)
    plt.tight_layout()
    plt.show()

    # ---- Boxplots ----
    # A boxplot shows the median (middle line), the interquartile
    # range (the box), and outliers (dots beyond the "whiskers").
    # Great for spotting outliers at a glance. We only plot the first
    # 10 numeric columns so the chart doesn't become unreadable when
    # a dataset has 50+ features.
    plt.figure(figsize=(18, 6))
    sns.boxplot(data=df.select_dtypes(include=np.number).iloc[:, :min(10, len(numeric_cols))])
    plt.xticks(rotation=90)
    plt.title("Boxplots")
    plt.show()

    # ---- Correlation heatmap ----
    # Correlation measures how strongly two numeric columns move
    # together, from -1 (perfectly opposite) to +1 (perfectly
    # together), with 0 meaning no linear relationship. This heatmap
    # helps us spot features that are strongly related to each other
    # (redundant info) or strongly related to the target (useful
    # predictors).
    plt.figure(figsize=(15, 12))
    sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
    plt.title("Correlation Heatmap")
    plt.show()

    # ==============================================================
    # STEP 6: SCALE FEATURES
    # ==============================================================
    # Why scale? Many ML algorithms (KNN, Naive Bayes with continuous
    # features, gradient-based models) work by measuring DISTANCES or
    # relying on the raw magnitude of numbers. If one feature ranges
    # 0-1 and another ranges 0-100000, the second feature will
    # completely dominate the distance calculation just because its
    # numbers are bigger — not because it's actually more important.
    # Scaling puts every feature on the same footing.
    #
    # MinMaxScaler specifically squashes every feature into [0, 1]
    # using the formula: (x - min) / (max - min)
    scaler = None
    if scale:
        scaler = MinMaxScaler()
        X = scaler.fit_transform(X)
        # fit_transform() does two things at once:
        #   fit  -> learns the min and max of each column from X
        #   transform -> actually applies the (x-min)/(max-min) formula
    else:
        X = X.values  # convert DataFrame to a plain numpy array

    # ==============================================================
    # STEP 7: TRAIN / TEST SPLIT
    # ==============================================================
    # We can't test a model on the same data it was trained on — the
    # model could just "memorize" the answers and look artificially
    # perfect. So we hold back a chunk of the data (test_size, e.g.
    # 20%) that the model NEVER sees during training, and only use it
    # at the very end to check how well the model generalizes to new,
    # unseen data.
    #
    # stratify=y is important for classification: it makes sure the
    # train set and test set both keep roughly the SAME proportion of
    # each class as the original data. Without it, a random split
    # could accidentally put almost all of one class into the test
    # set, giving a misleading evaluation.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print("\nTraining Shape :", X_train.shape)
    print("Testing Shape  :", X_test.shape)
    print("\nTraining Class Distribution")
    print(y_train.value_counts())
    print("\nTesting Class Distribution")
    print(y_test.value_counts())

    # ==============================================================
    # RETURN everything downstream code will need, bundled in a dict
    # so we don't have to keep passing around 8 separate variables.
    # ==============================================================
    return {
        "df": df,
        "X": X,
        "y": y,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler
    }

### Usage — Assignment 1

Swap `"spambase_updated.csv"` and `"spam"` for any other dataset/target and it works the same way.

In [ ]:
df = pd.read_csv("Spambase_Dataset.csv")

data = assn1_preprocess(df, "spam")

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]
X = data["X"]
y = data["y"]
df = data["df"]